# NanoFTIR interferogram processing workflow

This notebook helps you to reprocess interferograms from nanoFTIR measurements.

## Import neccesarry packages

In [1]:
import numpy as np
import copy
from glob import glob
import os

import os, sys
parent_dir = os.path.dirname(os.getcwd())
sys.path.append(parent_dir)

import numpy as np
import pySNOM
from pySNOM import readers
from pySNOM import spectra
from pySNOM.interferograms import NeaInterferogram, ProcessSingleChannel, ProcessMultiChannels, ProcessAllPoints

import plotly.graph_objects as go
from plotly.subplots import make_subplots


### Read the interferograms

In [26]:
# fdata = 'datasets/testifg_singlepoint.txt'
# data_reader = readers.NeaInterferogramReader(os.path.join(pySNOM.__path__[0], fdata))

onedrive_folder = glob(os.path.expanduser("~\\OneDrive"))
print(onedrive_folder)
basefolder = os.path.join(onedrive_folder[0], r"SOLEIL\Projects\Users\Gregory Stoclet")

fname_ref = '2025-07-02 165418 Interferograms SuperGold-reference.txt'
fname_sample = '2025-07-02 183646 Interferograms BR1_100_pointdroite.txt'
fname_sample = '2025-07-02 182325 Interferograms BR1_100_pointgauche.txt'
# fname_sample = '2025-07-02 181006 Interferograms BR1_100_pointhaut.txt'

data_reader_ref = readers.NeaSpectralReader(os.path.join(basefolder, fname_ref))
data, measparams = data_reader_ref.read()
ifgr = NeaInterferogram(data,measparams,filename=fname_ref)

data_reader_sample = readers.NeaSpectralReader(os.path.join(basefolder, fname_sample))
data, measparams = data_reader_sample.read()
ifgs = NeaInterferogram(data,measparams,filename=fname_sample)

print(ifgr.data.keys())

['C:\\Users\\ngerg\\OneDrive']
dict_keys(['Row', 'Column', 'Run', 'Depth', 'Z', 'M', 'O0A', 'O0P', 'O1A', 'O1P', 'O2A', 'O2P', 'O3A', 'O3P', 'O4A', 'O4P', 'O5A', 'O5P'])


In [27]:
a = np.reshape(
        ifgr.data["O2A"],
        (ifgr.parameters["Averaging"], ifgr.parameters["PixelArea"][2]),
    )
m = np.reshape(
        ifgr.data["M"],
        (ifgr.parameters["Averaging"], ifgr.parameters["PixelArea"][2]),
    )

z = np.reshape(
        ifgr.data["Z"],
        (ifgr.parameters["Averaging"], ifgr.parameters["PixelArea"][2]),
    )

for i in range(np.shape(a)[0]):
    a[i] = a[i]-np.median(a[i])

# Plotting
# fig = go.Figure()

# xrange = [650, 2000]
# yrange = [-0.8,0.5]

# xdata = m
# ydata = a
# fig.add_trace(go.Scatter(x=xdata,y=ydata, mode='markers'))

# fig.update_layout(legend_title_text = "Spectra")
# fig.update_xaxes(title_text="Mirror position / mm")
# fig.update_yaxes(title_text="Value")
# fig.show()

xaxis = np.arange(0, np.shape(a)[1])
yaxis = np.arange(0, np.shape(a)[0])

fig = make_subplots(rows=2, cols=1, subplot_titles=(["random corrected image"]))
fig.add_trace(go.Heatmap(
        x = xaxis,
        y = yaxis,
        z = a,
        zmin=-1, zmax=1),
        row=1, col=1)

idx = 472

t = pySNOM.images.RemoveSpikes(threshold=0.6, absolute_threshold=True, method='laplace', higher=True)
cc = t.transform(copy.deepcopy(a[:,0:idx]))
t = pySNOM.images.RemoveSpikes(threshold=-1, absolute_threshold=True, method='laplace', higher=False)
cc = t.transform(cc)

na = copy.deepcopy(a)
na[:,0:idx] = copy.deepcopy(cc)

fig.add_trace(go.Heatmap(
        x = xaxis,
        y = yaxis,
        z = na,
        zmin=-1, zmax=1),
        row=2, col=1)
fig.show()

In [20]:
ifg_bad = na[23,:]
ifg_good = na[26,:]
noises = np.std(cc, axis=1)

# Check the FFT
fft_bad, wn_bad = pySNOM.interferograms.ProcessInterferogram(apod=False).transform(ifg_bad,m[24,:])
fft_good, wn_good = pySNOM.interferograms.ProcessInterferogram(apod=False).transform(ifg_good,m[26,:])

fig = go.Figure()
# fig.add_trace(go.Scatter(y=noises))
# fig.add_trace(go.Scatter(y=ifg_bad, mode='lines', name='bad ifg'))
# fig.add_trace(go.Scatter(y=ifg_good, mode='lines', name='good ifg'))
fig.add_trace(go.Scatter(y=np.abs(fft_bad), mode='lines', name='bad ifg'))
fig.add_trace(go.Scatter(y=np.abs(fft_good), mode='lines', name='good ifg'))

### Check Laplace filling spike removal with single IFG

In [24]:
ifgtest = a[52,:472].copy()
t = pySNOM.images.RemoveSpikes(threshold=0.5, absolute_threshold=True, method='median', higher=True)
ifgtest = t.transform(ifgtest)

fig = go.Figure()
fig.add_trace(go.Scatter(y=ifgtest))
fig.add_trace(go.Scatter(y=a[52,:472]))

In [ ]:
spikey_idx = 52
little_jump_idx = 22
valuedift_idx = 23
jumpy_idx = 24



In [ ]:
fig = make_subplots(rows=2, cols=1)

fig.add_trace(go.Scatter(y=a[52,:472]))
fig.add_trace(go.Scatter(y=z[52,:472]))